In [ ]:
!lscpu

In [ ]:
!nvidia-smi

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"cuDNN version: {torch.backends.cudnn.version()}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path(os.environ.get(
    "AUV_PROJECT_DIR",
    "/content/drive/MyDrive/Colab Notebooks/auvhamnode/g3_5_7",
))
assert PROJECT_DIR.exists(), f"Project directory not found: {PROJECT_DIR}"
%cd $PROJECT_DIR

In [ ]:
%pip install -q torchdiffeq pandas

# Phase-1A OC v4-lite formal workflow

**定位**：当前 Phase-1A 的正式 Colab 执行入口。

**目标**：干净执行 OC-only 三模型五 seed protocol-sensitivity check，判断 trajectory-consistent `v4_lite` noisy IC 是否改变当前 `iid_noisy_ic` 下的 PHNODE / structured dynamics 结论。

**边界**：

- 只跑 `oc + known-current surrogate`。
- 不复用旧 checkpoint / suite / proxy。
- 不生成旧 `phase1_*` 兼容产物。
- 不默认运行 Phase-1B 模型，也不默认运行 `degraded_eval` / `heading_biased_eval`。
- 每个长任务 cell 直接调用 `scripts/run_phase1a_oc_v4lite.sh`，保留实时 stdout/stderr。

## 0. Shared configuration

默认输出：

- training suites: `checkpoints/sweep_oc_phase1a_{smoke1,smoke3,decision}_{clean,iid,v4lite}_${RUN_TAG}`
- local proxy suites: `/content/_proxy_suites/sweep_oc_phase1a_{smoke1,smoke3,decision}_proxy_${RUN_TAG}`
- exported decision artifacts: `checkpoints/sweep_oc_phase1a_decision_proxy_${RUN_TAG}/phase1a_*`
- execution logs: `checkpoints/phase1a_logs/${RUN_TAG}/`
- run metadata: `checkpoints/phase1a_metadata_${RUN_TAG}/phase1a_{run_config,environment}.json`

如果需要重跑同一文档，先换 `RUN_TAG`。`preflight` 会拒绝覆盖已存在的目标目录。

In [ ]:
import os

# Runtime
os.environ["PYTHON_BIN"] = "python"
os.environ["DEVICE"] = "cuda"
os.environ["LOCAL_PROXY_ROOT"] = "/content/_proxy_suites"

# Phase-1A identity
os.environ["RUN_TAG"] = "phase1a_oc_v4lite_cleanrun_v1"
os.environ["DATASET"] = "data/auv_oc_traj1000_blk150_s23_d0be9434.pkl"
os.environ["NOISE_REFERENCE"] = "remus100_dr"
os.environ["PHASE1A_LOG_DIR"] = str(PROJECT_DIR / "checkpoints" / "phase1a_logs" / os.environ["RUN_TAG"])
os.environ["PHASE1A_METADATA_DIR"] = str(PROJECT_DIR / "checkpoints" / f"phase1a_metadata_{os.environ['RUN_TAG']}")

# Phase-1A matrix
os.environ["PHASE1A_MODELS"] = "phnode_full ablate_no_mass_prior ablate_no_lift"
os.environ["SMOKE1_MODELS"] = "phnode_full"
os.environ["SMOKE_SEEDS"] = "42 44 46"
os.environ["DECISION_SEEDS"] = "42 43 44 45 46"

# Evaluation contract
os.environ["SMOKE_EVAL_NUM_TRAJ_PER_SCENARIO"] = "6"
os.environ["DECISION_EVAL_NUM_TRAJ_PER_SCENARIO"] = "30"
os.environ["EVAL_TIMES"] = "10 30 60"
os.environ["EVAL_SCENARIOS"] = "PRBS CHIRP OU"
os.environ["EVAL_BASE_SEED"] = "42"
os.environ["EVAL_NOISE_SEED"] = "2024"
os.environ["EVAL_PROGRESS_EVERY"] = "5"
os.environ["EVAL_NUM_DIAGNOSTIC_PLOTS"] = "6"
os.environ["IID_EVAL_PROFILES"] = "clean nominal_eval"
os.environ["V4_EVAL_PROFILES"] = "nominal_eval"

# Audit gate
os.environ["STRICT_ZERO_NOISE_AUDIT"] = "1"
os.environ["SOFT_MIN_EPOCH_SCALE"] = "0.05"

print("RUN_TAG=", os.environ["RUN_TAG"])
print("PHASE1A_MODELS=", os.environ["PHASE1A_MODELS"])
print("DECISION_SEEDS=", os.environ["DECISION_SEEDS"])
print("PHASE1A_LOG_DIR=", os.environ["PHASE1A_LOG_DIR"])
print("PHASE1A_METADATA_DIR=", os.environ["PHASE1A_METADATA_DIR"])

## 1. Preflight

确认本次 clean run 的所有目标 suite / proxy 目录都不存在，并保存本次 run config / environment metadata。若这里失败，换 `RUN_TAG` 或手动清理目标目录。

In [ ]:
os.environ["MODE"] = "preflight"
!bash scripts/run_phase1a_oc_v4lite.sh

## 2. Smoke-1 train + protocol validation

最小单模型 gate：`phnode_full × seeds 42/44/46 × clean/iid/v4lite`。

本 cell 完成训练、训练审计和 mandatory `v4_lite` protocol validation。

In [ ]:
os.environ["MODE"] = "smoke1_train"
!bash scripts/run_phase1a_oc_v4lite.sh

## 3. Smoke-1 rollout eval + local proxy

使用小 rollout 数检查 eval、summary 和 report 路径。Smoke 只验证流程，不写研究结论。

In [ ]:
os.environ["MODE"] = "smoke1_eval"
!bash scripts/run_phase1a_oc_v4lite.sh

## 4. Smoke-3 train + protocol validation

三模型 smoke：`phnode_full / ablate_no_mass_prior / ablate_no_lift × seeds 42/44/46`。

In [ ]:
os.environ["MODE"] = "smoke3_train"
!bash scripts/run_phase1a_oc_v4lite.sh

## 5. Smoke-3 rollout eval + local proxy

通过后再进入五 seed decision run。

In [ ]:
os.environ["MODE"] = "smoke3_eval"
!bash scripts/run_phase1a_oc_v4lite.sh

## 6. Decision train + protocol validation

正式 Phase-1A decision matrix：三模型 × seeds `42/43/44/45/46` × `clean/iid_noisy_ic/v4_lite`。

In [ ]:
os.environ["MODE"] = "decision_train"
!bash scripts/run_phase1a_oc_v4lite.sh

## 7. Decision rollout eval

默认只跑 Phase-1A 必需评估：

- `clean`
- `iid_noisy_ic / nominal_eval`
- `v4_lite / nominal_eval`

`degraded_eval` 和 `heading_biased_eval` 不在默认 gate 内。

In [ ]:
os.environ["MODE"] = "decision_eval"
!bash scripts/run_phase1a_oc_v4lite.sh

## 8. Decision summarize + export

注册 decision proxy suite，生成并导出 `phase1a_*` 正式产物。

In [ ]:
os.environ["MODE"] = "decision_summarize"
!bash scripts/run_phase1a_oc_v4lite.sh

## 9. Inspect Phase-1A artifacts

先看 model-level summary，再回到 by-seed / degradation 检查收益是否主要来自单个坏 seed。

In [ ]:
from pathlib import Path
import os
import pandas as pd
from IPython.display import Markdown, display

out = Path("checkpoints") / f"sweep_oc_phase1a_decision_proxy_{os.environ['RUN_TAG']}"
print(out)

summary = pd.read_csv(out / "phase1a_summary.csv")
by_seed = pd.read_csv(out / "phase1a_by_seed.csv")
by_horizon = pd.read_csv(out / "phase1a_by_horizon.csv")
degradation = pd.read_csv(out / "phase1a_degradation.csv")
protocol_delta = pd.read_csv(out / "phase1a_protocol_delta.csv")

display(summary.head(20))
display(by_seed.head(20))
display(by_horizon.head(20))
display(degradation.head(20))
display(protocol_delta.head(20))

In [ ]:
brief = Path("checkpoints") / f"sweep_oc_phase1a_decision_proxy_{os.environ['RUN_TAG']}" / "phase1a_decision_brief.md"
display(Markdown(brief.read_text()))

## 10. Optional diagnostics

`degraded_eval` 和 `heading_biased_eval` 仅作为后续诊断或 Phase-1B 条件扩展，不在本 notebook 的默认 clean run 中执行。